# Entrenar y evaluar YOLO para Área Técnica

En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, elige una GPU. Ten a mano **worker-entrenamiento-colab.zip**, **dataset-futbol-yolo.zip** y **prueba-integral-10s.mp4** (o un clip de prueba con dimensiones y colores conocidos). Ejecuta las celdas en orden.

El cuaderno guarda los pesos, la comparación sobre las imágenes anotadas y las muestras que produciría el worker al procesar el clip como grabación y como directo simulado. El ensayo de red, cámara y visor se realiza en la aplicación.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Selecciona un acelerador GPU y vuelve a ejecutar esta celda.")
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
import sys

raiz = Path(mkdtemp(prefix="area-tecnica-", dir="/content"))
print("Selecciona los dos ZIP y un archivo MP4.")
subidos = files.upload()
rutas = [Path(nombre).resolve() for nombre in subidos]
subidos.clear()
paquete = [ruta for ruta in rutas if ruta.name == "worker-entrenamiento-colab.zip"]
datos = [ruta for ruta in rutas if ruta.name == "dataset-futbol-yolo.zip"]
videos = [ruta for ruta in rutas if ruta.suffix.lower() == ".mp4"]
if not (len(paquete) == len(datos) == len(videos) == 1):
    raise ValueError("Selecciona worker-entrenamiento-colab.zip, dataset-futbol-yolo.zip y un MP4.")
video = videos[0]
dataset = raiz / "dataset"
for fuente, destino in ((paquete[0], raiz), (datos[0], dataset)):
    with ZipFile(fuente) as archivo:
        if any(not (destino / nombre).resolve().is_relative_to(destino) for nombre in archivo.namelist()):
            raise ValueError("El ZIP contiene una ruta fuera de su carpeta de destino.")
        archivo.extractall(destino)
sys.path.insert(0, str(raiz / "vision"))

In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", str(raiz / "vision/requirements.txt")], check=True)

In [ ]:
from entrenamiento.entrenar import entrenar

epochs = 40
tamano_img = 960
batch = 4
pesos = entrenar(epochs=epochs, imgsz=tamano_img, batch=batch, device="0",
                dataset_yaml=dataset / "data.yaml").resolve()
print("Pesos del entrenamiento:", pesos)

### Guardar los pesos antes de continuar

Ejecuta la siguiente celda para descargar `best.pt` a tu computadora. Después continúa con la comparación.

In [ ]:
files.download(str(pesos))

In [ ]:
from deteccion.modelo import CLASES_COCO, CLASES_PROPIO, ConfigModelo, ModeloDeteccion
from evaluacion.validacion_yolo import validar

candidatos = {
    "modelo-actual": (Path("yolo11n.pt").resolve(), CLASES_COCO),
    "modelo-afinado": (pesos, CLASES_PROPIO),
}
imagenes = sorted((dataset / "images/val").glob("*.jpg"))
ap50 = {}
for nombre, (checkpoint, clases) in candidatos.items():
    config = ConfigModelo(ruta_pesos=str(checkpoint), dispositivo="cuda:0",
                         tamano_img=tamano_img, clases=clases)
    ap50[nombre] = validar(ModeloDeteccion(config), imagenes, dataset / "labels/val", CLASES_PROPIO)
print("AP50 por clase sobre", len(imagenes), "imágenes")
for nombre, metricas in ap50.items():
    print(nombre, metricas)

In [ ]:
import gc
import json
import shutil

from calibracion.instalar_modelo import instalar
from evaluacion.ensayo_video import ensayar

print("Modelo de cancha verificado:", instalar())
segundos = 15
calibracion = {"modo": "AUTOMATICA", "largo_m": 105, "ancho_m": 68}  # Ajustar al campo del clip.
color_propio, color_rival = "#C8102E", "#17202A"  # Ajustar a las camisetas del clip.
salida = Path(mkdtemp(prefix="resultados-", dir=raiz))
resumen = {}
for nombre, (checkpoint, clases) in candidatos.items():
    resumen[nombre] = {}
    for modo, directo in (("grabacion", False), ("directo-simulado", True)):
        config = ConfigModelo(ruta_pesos=str(checkpoint), dispositivo="cuda:0",
                             tamano_img=tamano_img, clases=clases)
        modelo = ModeloDeteccion(config)
        resultado = ensayar(modelo, video, salida / nombre / modo, segundos=segundos,
                            simular_directo=directo, calibracion=calibracion,
                            color_propio=color_propio, color_rival=color_rival)
        if not str(resultado["dispositivo"]).startswith("cuda"):
            raise RuntimeError("El worker no utilizó CUDA.")
        resumen[nombre][modo] = resultado
        print(nombre, modo, resultado["cuadros_inferidos"], "cuadros;",
              resultado["muestras_con_calibracion_valida"], "muestras con referencia")
        del modelo
        gc.collect()
        torch.cuda.empty_cache()

shutil.copy2(pesos, salida / "best.pt")
shutil.copy2(dataset / "manifiesto.json", salida / "dataset-origen.json")
(salida / "comparacion.json").write_text(json.dumps({
    "imagenes_validacion": len(imagenes), "AP50": ap50, "epochs": epochs,
    "tamano_img": tamano_img, "gpu": torch.cuda.get_device_name(0), "ensayos": resumen,
}, indent=2, ensure_ascii=False), encoding="utf-8")
zip_resultados = shutil.make_archive(str(raiz / "resultado-yolo"), "zip", salida)
print(zip_resultados)

In [ ]:
files.download(zip_resultados)

**Cómo leer los resultados**. Compara el AP50 de jugadores y balón sobre las mismas imágenes. El modelo COCO no distingue al árbitro. En el clip, revisa las cajas, la continuidad del seguimiento y los cambios de calibración en `muestras.json`. Los cuadros con alguna detección de balón miden cobertura, y la precisión requiere anotaciones del clip. Los tiempos del directo simulado incluyen el reloj del archivo.